# 37. Neural Networks: Transformers

## Algorithm Category
**Type**: Neural Networks - Attention-Based  
**Complexity**: Very High  
**Use Case**: Natural language processing, sequence-to-sequence tasks, large language models

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the Transformer architecture and self-attention mechanism
- Implement attention mechanisms from scratch
- Understand encoder-decoder architecture
- Apply Transformers to sequence tasks
- Visualize attention weights
- Understand positional encoding

## Historical Context

Transformers were introduced by Vaswani et al. in 2017:
- Vaswani, A., et al. (2017): "Attention is All You Need"
- Revolutionized NLP and sequence modeling
- Foundation for BERT, GPT, and modern LLMs

**Key Papers/References:**
- Vaswani, A., et al. (2017). "Attention is All You Need"
- Devlin, J., et al. (2018). "BERT: Pre-training of Deep Bidirectional Transformers"

## When to Use Transformers

Transformers are appropriate when:
- Natural language processing tasks
- Long-range dependencies needed
- Parallel processing is important
- Sequence-to-sequence tasks
- When attention mechanism is beneficial
- Large-scale language modeling

## Theory & Mechanics

### Mathematical Foundation

**Self-Attention:**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Where:
- $Q$: Query matrix
- $K$: Key matrix
- $V$: Value matrix
- $d_k$: Dimension of keys

**Multi-Head Attention:**
$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$$

Where each head is:
$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

**Positional Encoding:**
$$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d_{model}})$$
$$PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d_{model}})$$

### Key Components

1. **Self-Attention**
   - Computes relationships between all positions
   - Allows parallel processing
   - Captures long-range dependencies
   - No recurrence needed

2. **Multi-Head Attention**
   - Multiple attention mechanisms in parallel
   - Captures different types of relationships
   - Increases model capacity

3. **Positional Encoding**
   - Adds position information to embeddings
   - Enables understanding of sequence order
   - Sinusoidal or learned

4. **Encoder-Decoder Architecture**
   - Encoder: Processes input sequence
   - Decoder: Generates output sequence
   - Attention connects encoder and decoder

### How It Works

1. **Input Embedding**: Convert tokens to vectors
2. **Add Positional Encoding**: Include position info
3. **Encoder**: Multi-head self-attention + FFN
4. **Decoder**: Masked self-attention + encoder-decoder attention
5. **Output**: Generate predictions

### Key Hyperparameters

- **d_model**: Model dimension
- **n_heads**: Number of attention heads
- **n_layers**: Number of encoder/decoder layers
- **d_ff**: Feed-forward dimension
- **dropout**: Regularization rate
- **max_seq_length**: Maximum sequence length

### Advantages

- Parallel processing (faster than RNNs)
- Long-range dependencies
- State-of-the-art NLP performance
- Transfer learning (pre-trained models)
- Attention interpretability

### Limitations

- Quadratic complexity with sequence length
- Requires large datasets
- Computationally expensive
- Memory intensive
- Hard to interpret fully


## Implementation

Let's implement a simplified Transformer for sequence classification.


In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import math

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print("Libraries imported successfully!")


In [ ]:
# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super(PositionalEncoding, self).__init__()
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# Multi-Head Self-Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % n_heads == 0
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attention_weights = torch.softmax(scores, dim=-1)
        output = torch.matmul(attention_weights, V)
        return output, attention_weights
    
    def forward(self, x, mask=None):
        batch_size = x.size(0)
        
        Q = self.W_q(x).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        attn_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        return self.W_o(attn_output), attention_weights

print("Positional Encoding and Multi-Head Attention defined!")


In [ ]:
# Transformer Encoder Block
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super(TransformerBlock, self).__init__()
        self.attention = MultiHeadAttention(d_model, n_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Self-attention with residual connection
        attn_output, _ = self.attention(x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        
        # Feed-forward with residual connection
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        
        return x

# Simple Transformer for Classification
class SimpleTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=2, d_ff=512, max_len=100, num_classes=2):
        super(SimpleTransformer, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)
        ])
        
        self.classifier = nn.Linear(d_model, num_classes)
    
    def forward(self, x, mask=None):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        
        for transformer in self.transformer_blocks:
            x = transformer(x, mask)
        
        # Global average pooling
        x = x.mean(dim=1)
        return self.classifier(x)

print("Transformer model defined!")


In [ ]:
# Create simple sequence data for demonstration
def create_sequence_data(n_samples=500, seq_length=20, vocab_size=50):
    """Create synthetic sequence classification data"""
    X = []
    y = []
    
    for _ in range(n_samples):
        # Random sequence
        seq = np.random.randint(1, vocab_size, seq_length)
        X.append(seq)
        
        # Simple classification: sum > threshold
        label = 1 if seq.sum() > seq_length * vocab_size / 2 else 0
        y.append(label)
    
    return np.array(X), np.array(y)

# Generate data
X, y = create_sequence_data(n_samples=500, seq_length=20, vocab_size=50)

# Split data
split_idx = int(0.8 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Convert to tensors
X_train_tensor = torch.LongTensor(X_train).to(device)
X_test_tensor = torch.LongTensor(X_test).to(device)
y_train_tensor = torch.LongTensor(y_train).to(device)
y_test_tensor = torch.LongTensor(y_test).to(device)

print(f"Training sequences: {X_train.shape}")
print(f"Test sequences: {X_test.shape}")
print(f"Vocabulary size: 50")
print(f"Number of classes: 2")


## Training

Let's train the Transformer model.


In [ ]:
# Initialize model
vocab_size = 50
d_model = 128
n_heads = 4
n_layers = 2
d_ff = 512

model = SimpleTransformer(
    vocab_size=vocab_size,
    d_model=d_model,
    n_heads=n_heads,
    n_layers=n_layers,
    d_ff=d_ff,
    num_classes=2
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Training loop
num_epochs = 20
train_losses = []
train_accuracies = []

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    
    # Calculate accuracy
    _, predicted = torch.max(outputs.data, 1)
    accuracy = (predicted == y_train_tensor).float().mean().item()
    
    train_losses.append(loss.item())
    train_accuracies.append(accuracy)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}: Loss = {loss.item():.4f}, Accuracy = {accuracy:.3f}")

print("\nTraining complete!")


## Evaluation

Let's evaluate the model and visualize attention.


In [ ]:
# Evaluate on test set
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, test_predicted = torch.max(test_outputs.data, 1)
    test_accuracy = (test_predicted == y_test_tensor).float().mean().item()

print(f"Test Accuracy: {test_accuracy:.3f}")

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, 'o-')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(train_accuracies, 's-', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Visualize attention weights (from first attention head)
model.eval()
sample_input = X_test_tensor[:1]

# Hook to capture attention weights
attention_weights_list = []

def attention_hook(module, input, output):
    if len(output) == 2:
        attn_output, attn_weights = output
        attention_weights_list.append(attn_weights.detach().cpu().numpy())

# Register hook on first transformer block
hook = model.transformer_blocks[0].attention.register_forward_hook(attention_hook)

with torch.no_grad():
    _ = model(sample_input)

hook.remove()

if attention_weights_list:
    attn_weights = attention_weights_list[0][0, 0]  # First head, first batch
    
    plt.figure(figsize=(10, 8))
    plt.imshow(attn_weights, cmap='viridis', aspect='auto')
    plt.colorbar(label='Attention Weight')
    plt.xlabel('Key Position')
    plt.ylabel('Query Position')
    plt.title('Attention Weights (First Head, First Layer)')
    plt.tight_layout()
    plt.show()
    
    print("Attention weights shape:", attn_weights.shape)
    print("Attention weights show which positions the model focuses on.")


## Validation & Testing

Let's validate the model.


In [ ]:
# Assertions
assert test_accuracy > 0.5, "Transformer should perform better than random"
assert len(train_losses) == num_epochs, "Should have trained for all epochs"
print("\n✓ Validation checks passed")

print("\nNote: Transformers have revolutionized NLP and sequence modeling.")
print("Key advantages: parallel processing, long-range dependencies,")
print("and state-of-the-art performance on many tasks.")


## Summary & Key Takeaways

### Key Concepts Learned

1. **Self-Attention Mechanism**
   - Computes relationships between all positions
   - Allows parallel processing
   - Captures long-range dependencies
   - No recurrence needed

2. **Multi-Head Attention**
   - Multiple attention mechanisms in parallel
   - Captures different types of relationships
   - Increases model capacity
   - Enables richer representations

3. **Positional Encoding**
   - Adds position information to embeddings
   - Enables understanding of sequence order
   - Sinusoidal or learned embeddings
   - Critical for sequence understanding

4. **Transformer Architecture**
   - Encoder: Processes input sequence
   - Decoder: Generates output sequence
   - Stack of transformer blocks
   - Residual connections and layer normalization

### When to Use Transformers

✅ **Good for:**
- Natural language processing
- Long-range dependencies
- When parallel processing is important
- Sequence-to-sequence tasks
- Large-scale language modeling
- Transfer learning (pre-trained models)

❌ **Not ideal for:**
- Very short sequences (overhead)
- When quadratic complexity is prohibitive
- Small datasets (need pre-training)
- Real-time applications (can be slow)
- When interpretability is critical

### Next Steps

- Explore **BERT** (Bidirectional Encoder Representations)
- Try **GPT** (Generative Pre-trained Transformer)
- Use **Hugging Face Transformers** library
- Apply to **machine translation** tasks
- Experiment with **fine-tuning** pre-trained models
